# 02 - Data Preprocessing & Splitting
## Proyek: RoadDamage_AI

### Tujuan Preprocessing
1. Memvalidasi integritas pasangan gambar dan label.
2. Membagi dataset ke dalam 3 subset:
   - **Train Set (70%):** Untuk proses pelatihan bobot YOLOv8n.
   - **Validation Set (20%):** Untuk evaluasi performa dan tuning hiperparameter selama training.
   - **Test Set (10%):** Untuk pengujian final (*unseen data*).
3. Menata struktur folder YOLO standar: `data/train`, `data/val`, `data/test`.
4. Membuat file konfigurasi `data/data.yaml` untuk Ultralytics YOLOv8.


In [1]:
import os
import glob
import shutil
import random
import yaml
from collections import Counter

DATA_DIR = os.path.abspath("../data")
IMAGES_DIR = os.path.join(DATA_DIR, "images")
LABELS_DIR = os.path.join(DATA_DIR, "labels-YOLO")

print(f"Data Root: {DATA_DIR}")


Data Root: z:\Projects\RoadDamage_AI\data


---
### 1. Mengumpulkan Seluruh Pasangan Data Valid


In [2]:
all_images = sorted(glob.glob(os.path.join(IMAGES_DIR, "*.jpg")))
print(f"Total Gambar Ditemukan: {len(all_images)}")

valid_pairs = []
for img_p in all_images:
    base = os.path.splitext(os.path.basename(img_p))[0]
    lbl_p = os.path.join(LABELS_DIR, f"{base}.txt")
    if os.path.exists(lbl_p):
        valid_pairs.append((img_p, lbl_p, base))

print(f"Total Pasangan Valid: {len(valid_pairs)}")


Total Gambar Ditemukan: 2009
Total Pasangan Valid: 2009


---
### 2. Melakukan Pembagian Dataset (Train 70%, Val 20%, Test 10%)
Kita menggunakan fixed random seed (`seed=42`) agar pembagian dataset bersifat **reproducible**.


In [3]:
random.seed(42)
random.shuffle(valid_pairs)

n_total = len(valid_pairs)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.20)
n_test = n_total - n_train - n_val

train_set = valid_pairs[:n_train]
val_set = valid_pairs[n_train:n_train + n_val]
test_set = valid_pairs[n_train + n_val:]

print(f"Train Set : {len(train_set)} gambar ({len(train_set)/n_total*100:.1f}%)")
print(f"Val Set   : {len(val_set)} gambar ({len(val_set)/n_total*100:.1f}%)")
print(f"Test Set  : {len(test_set)} gambar ({len(test_set)/n_total*100:.1f}%)")


Train Set : 1406 gambar (70.0%)
Val Set   : 401 gambar (20.0%)
Test Set  : 202 gambar (10.1%)


---
### 3. Membuat Struktur Direktori Split YOLO


In [4]:
splits = {
    "train": train_set,
    "val": val_set,
    "test": test_set
}

for split_name, split_items in splits.items():
    img_dest = os.path.join(DATA_DIR, split_name, "images")
    lbl_dest = os.path.join(DATA_DIR, split_name, "labels")
    os.makedirs(img_dest, exist_ok=True)
    os.makedirs(lbl_dest, exist_ok=True)
    
    for img_p, lbl_p, base in split_items:
        d_img = os.path.join(img_dest, f"{base}.jpg")
        d_lbl = os.path.join(lbl_dest, f"{base}.txt")
        if not os.path.exists(d_img):
            shutil.copyfile(img_p, d_img)
        if not os.path.exists(d_lbl):
            shutil.copyfile(lbl_p, d_lbl)

print("Folder split train, val, dan test berhasil dibuat dan diverifikasi!")


Folder split train, val, dan test berhasil dibuat dan diverifikasi!


---
### 4. Membuat Konfigurasi `data/data.yaml`


In [5]:
data_yaml_data = {
    "path": DATA_DIR.replace("\\", "/"),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 3,
    "names": ["pothole", "crack", "manhole"]
}

yaml_file_path = os.path.join(DATA_DIR, "data.yaml")
with open(yaml_file_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml_data, f, default_flow_style=False, sort_keys=False)

print(f"data.yaml berhasil dibuat di: {yaml_file_path}")

with open(yaml_file_path, "r", encoding="utf-8") as f:
    print("\nIsi data.yaml:")
    print(f.read())


data.yaml berhasil dibuat di: z:\Projects\RoadDamage_AI\data\data.yaml

Isi data.yaml:
path: z:/Projects/RoadDamage_AI/data
train: train/images
val: val/images
test: test/images
nc: 3
names:
- pothole
- crack
- manhole



---
### 5. Kesimpulan Preprocessing
Dataset telah terbagi secara rapi ke dalam 3 subset independen dengan konfigurasi `data.yaml` yang siap dibaca oleh Ultralytics YOLOv8.
